# Notebook 06: Gradient Boosting and XGBoost

## Why This Matters

XGBoost, LightGBM, and CatBoost have dominated structured/tabular data competitions on Kaggle for over a decade and are the most commonly deployed ML models in industry for fraud detection, credit scoring, and ad ranking. Every serious ML interview for roles involving tabular data will ask you to explain gradient boosting, contrast XGBoost with vanilla GBM, and justify hyperparameter choices. XGBoost's innovations — second-order gradients, regularization terms in the tree objective, column subsampling, and the histogram algorithm — are canonical improvements that interviewers test directly. Understanding these details, not just calling `xgb.fit()`, is what differentiates strong candidates.

## 1. XGBoost: What Makes It Different from Vanilla GBM

Vanilla gradient boosting (sklearn GBM) uses first-order gradients only. XGBoost uses **second-order Taylor approximation** of the loss, which gives more precise gradient information and enables a closed-form analytical solution for the optimal leaf values.

**XGBoost objective at step m:**
$$\mathcal{L}^{(m)} = \sum_i [g_i f_m(x_i) + \frac{1}{2} h_i f_m(x_i)^2] + \Omega(f_m)$$

where $g_i = \partial L / \partial \hat{y}_i$ (first-order), $h_i = \partial^2 L / \partial \hat{y}_i^2$ (second-order), and
$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_j w_j^2$ is a regularization term on the tree structure (T = number of leaves, w = leaf values).

Key innovations vs. sklearn GBM:
1. **Second-order gradients**: more accurate gradient estimates → fewer trees needed
2. **Built-in L1/L2 regularization** on leaf weights (gamma, reg_alpha, reg_lambda)
3. **Column (feature) subsampling**: like Random Forest, reduces correlation between trees
4. **Row subsampling** (subsample): reduces overfitting
5. **Histogram algorithm**: bins continuous features for O(n → n_bins) split search speedup
6. **Sparsity-aware**: efficient handling of missing values and sparse features
7. **Parallel split finding**: evaluates all features in parallel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier

# Check if xgboost is available; fall back to sklearn GBM if not
try:
    import xgboost as xgb
    HAS_XGB = True
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed. Demonstrating with sklearn GradientBoostingClassifier.")
    print("Install with: pip install xgboost")

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Compare sklearn GBM vs XGBoost (if available)
results = {}

# Sklearn GBM
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                  subsample=0.8, random_state=42)
gbm.fit(X_train, y_train)
results['sklearn GBM'] = roc_auc_score(y_test, gbm.predict_proba(X_test)[:, 1])

if HAS_XGB:
    xgb_model = xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    results['XGBoost'] = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

for name, auc in results.items():
    print(f"{name:20s} | AUC = {auc:.4f}")

## 2. Second-Order Gradients: Why They Help

To build intuition, let's look at the Taylor approximation. For MSE loss $L = \frac{1}{2}(y - \hat{y})^2$:
- First-order gradient: $g_i = \hat{y}_i - y_i$ (the residual)
- Second-order gradient: $h_i = 1$ (constant for MSE)

For log-loss $L = -[y \log p + (1-y) \log(1-p)]$:
- $g_i = p_i - y_i$ (prediction minus label)
- $h_i = p_i(1-p_i)$ (variance of Bernoulli)

The hessian $h_i$ tells us the **curvature** of the loss at each point. When $h_i$ is large (near p=0.5), the loss is steep and we should take a bigger step. When $h_i$ is small (near p=0 or p=1), the prediction is already confident and we should barely adjust.

The optimal leaf value in XGBoost (minimizing the approximate objective) is:
$$w_j^* = -\frac{\sum_{i \in j} g_i}{\sum_{i \in j} h_i + \lambda}$$

This is a weighted average of gradients, with hessian-based weights — more curvature → more influence.

In [ ]:
# Visualize hessian (h_i = p*(1-p)) for log-loss
p = np.linspace(0.001, 0.999, 300)
hessian = p * (1 - p)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(p, hessian, 'steelblue', linewidth=2)
axes[0].set_xlabel('Predicted probability p')
axes[0].set_ylabel('Hessian h = p(1-p)')
axes[0].set_title('Log-Loss Hessian: Highest Near Decision Boundary')
axes[0].axvline(0.5, color='red', linestyle='--', label='Decision boundary')
axes[0].legend()

# Show that second-order gives better descent direction
z = np.linspace(-3, 3, 200)
# For a point with y=1, p=sigmoid(z)
p_vals = 1 / (1 + np.exp(-z))
gradient  = p_vals - 1  # g = p - y (y=1)
hess_vals = p_vals * (1 - p_vals)
first_order_step  = -gradient
second_order_step = -gradient / (hess_vals + 1e-6)

axes[1].plot(z, first_order_step,  'coral',    label='1st-order step (-g)', linewidth=2)
axes[1].plot(z, second_order_step, 'steelblue', label='2nd-order step (-g/h)', linewidth=2)
axes[1].set_xlabel('Score z = w·x')
axes[1].set_ylabel('Update magnitude')
axes[1].set_title('1st vs 2nd Order: Newton Step (y=1)')
axes[1].legend()
axes[1].set_ylim(-2, 4)

plt.tight_layout()
plt.show()

## 3. Key Hyperparameters and Tuning Strategy

XGBoost has many hyperparameters but only a handful have strong effects. The canonical tuning order:

1. **`n_estimators` + `learning_rate`**: Set learning_rate=0.05–0.1 and use early stopping to find the right number of trees. These are the most important.
2. **`max_depth`**: Controls tree complexity. For GBM, shallower (3–6) is usually better than for RF.
3. **`subsample`** + **`colsample_bytree`**: Row and column sampling. Similar to RF's feature subsampling. Start at 0.8.
4. **`min_child_weight`**: Minimum sum of hessian in a leaf. Prevents splits on very uncertain samples. Raise this if overfitting.
5. **`gamma` / `min_split_loss`**: Minimum gain required to make a split. Raise this for stronger regularization.
6. **`reg_alpha`** (L1) + **`reg_lambda`** (L2): Leaf weight regularization. Default: alpha=0, lambda=1.

**Early stopping** is the most practical way to set `n_estimators`: train on a hold-out set and stop when validation loss doesn't improve for N rounds.

In [ ]:
# Early stopping and learning curves with sklearn GBM (works without xgboost)
from sklearn.model_selection import validation_curve

# Learning rate effect on convergence speed vs final performance
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3, 1.0]
n_trees_eval = 200

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in [0.01, 0.05, 0.1, 0.3]:
    gbm_lr = GradientBoostingClassifier(
        n_estimators=n_trees_eval, learning_rate=lr,
        max_depth=3, random_state=42
    ).fit(X_train, y_train)
    # staged_predict_proba gives predictions after each tree
    test_scores = [
        roc_auc_score(y_test, pred[:, 1])
        for pred in gbm_lr.staged_predict_proba(X_test)
    ]
    axes[0].plot(range(1, n_trees_eval + 1), test_scores, label=f'lr={lr}')

axes[0].set_xlabel('Number of trees')
axes[0].set_ylabel('Test AUC')
axes[0].set_title('Learning Rate Effect on Convergence')
axes[0].legend()

# Max depth effect
for d in [1, 2, 3, 5, 8]:
    gbm_d = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1,
        max_depth=d, random_state=42
    ).fit(X_train, y_train)
    test_scores_d = [
        roc_auc_score(y_test, pred[:, 1])
        for pred in gbm_d.staged_predict_proba(X_test)
    ]
    axes[1].plot(range(1, 201), test_scores_d, label=f'depth={d}')

axes[1].set_xlabel('Number of trees')
axes[1].set_ylabel('Test AUC')
axes[1].set_title('Max Depth Effect on GBM Performance')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. LightGBM: Key Architectural Difference

**LightGBM** (Microsoft, 2017) introduced two key algorithmic improvements over XGBoost:

1. **GOSS (Gradient-based One-Side Sampling)**: keep all samples with large gradients (high error), randomly sample from low-gradient samples. Result: faster training while focusing computation on hard examples.

2. **EFB (Exclusive Feature Bundling)**: bundle mutually exclusive sparse features (features that rarely take non-zero values simultaneously) into a single feature. Reduces effective feature count without losing information.

3. **Leaf-wise tree growth** (vs. XGBoost's level-wise): expand the leaf with the highest loss reduction, not all leaves at the current depth. Faster convergence on complex problems but higher overfitting risk on small data.

In practice: LightGBM is 5-10x faster than XGBoost on large datasets and often achieves similar or better accuracy.

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGB = True
    print(f"LightGBM version: {lgb.__version__}")
    
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05,
        num_leaves=31, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    lgb_auc = roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1])
    print(f"LightGBM AUC: {lgb_auc:.4f}")

except ImportError:
    HAS_LGB = False
    print("LightGBM not installed. Install with: pip install lightgbm")
    print("Key difference from XGBoost: leaf-wise tree growth, GOSS sampling, EFB bundling.")

## 5. Feature Importance in Gradient Boosting

Gradient boosting models offer multiple feature importance metrics:

- **`weight`** (gain frequency): number of times a feature is used to split across all trees. Biased toward high-cardinality features.
- **`gain`**: total gain from all splits using that feature. More reliable than frequency.
- **`cover`**: total number of samples affected by splits on that feature.

All three can disagree significantly. For model interpretation, **SHAP values** are the gold standard (see XAI series). For feature selection, **permutation importance** is most reliable.

In [ ]:
from sklearn.inspection import permutation_importance

# Use sklearn GBM since it's always available
gbm_fi = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, random_state=42)
gbm_fi.fit(X_train, y_train)

# MDI feature importance
mdi_importance = gbm_fi.feature_importances_

# Permutation importance
perm = permutation_importance(gbm_fi, X_test, y_test,
                               n_repeats=10, random_state=42, scoring='roc_auc')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, scores, title, color in [
    (axes[0], mdi_importance, 'MDI Feature Importance', 'steelblue'),
    (axes[1], perm.importances_mean, 'Permutation Importance (AUC)', 'coral'),
]:
    idx = np.argsort(scores)[-15:]  # top 15
    ax.barh(range(len(idx)), scores[idx], color=color, edgecolor='k')
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx], fontsize=8)
    ax.set_title(title, fontsize=10)

plt.suptitle('GBM Feature Importance: MDI vs. Permutation', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Partial Dependence Plots

**Partial dependence plots (PDP)** show the marginal effect of one or two features on the predicted outcome, averaging over all other features. They answer: "holding everything else constant, how does the prediction change as I vary feature X?"

Limitation: PDPs assume feature independence, which can be misleading when features are correlated. ICE plots (Individual Conditional Expectation) show the same relationship for each individual sample and reveal heterogeneous effects.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Plot PDP for top 4 features by MDI
top4_idx = np.argsort(mdi_importance)[-4:]
print("Top 4 features:", [feature_names[i] for i in top4_idx])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
disp = PartialDependenceDisplay.from_estimator(
    gbm_fi, X_train,
    features=list(top4_idx),
    feature_names=feature_names,
    kind='both',  # both PDP and ICE lines
    ax=axes
)
plt.suptitle('Partial Dependence + ICE Plots — Top 4 Features', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Model | Gradient Order | Tree Growth | Key Regularization | Best For |
|-------|---------------|-------------|-------------------|----------|
| sklearn GBM | 1st order | Level-wise | subsample, max_depth | Baseline; easy to debug |
| XGBoost | 2nd order | Level-wise | gamma, reg_alpha/lambda + column sampling | Large datasets; robust tuning |
| LightGBM | 2nd order (GOSS) | Leaf-wise | num_leaves, min_child_samples | Very large datasets; fastest |
| CatBoost | 2nd order | Symmetric | Native categoricals; ordered boosting | High-cardinality categoricals |

| Hyperparameter | Effect | Typical Range |
|---------------|--------|---------------|
| learning_rate | Slower convergence → better generalization | 0.01–0.1 |
| n_estimators | More trees, set with early stopping | 100–2000 |
| max_depth | Interaction order; deeper = more bias capacity | 3–8 |
| subsample | Row sampling fraction per tree | 0.6–1.0 |
| colsample_bytree | Column sampling fraction per tree | 0.6–1.0 |
| min_child_weight | Min hessian sum in leaf; prevents overfit | 1–20 |